In [3]:
import os
import tensorflow as tf
from onnx_tf.backend import prepare

# Ensure you have onnx and onnx-tf installed:
# pip install onnx onnx-tf tensorflow

def convert_onnx_to_tflite(onnx_model_path, tflite_model_path):
    """
    Converts an ONNX model to a TensorFlow Lite model.

    Args:
        onnx_model_path (str): The file path to the input ONNX model.
        tflite_model_path (str): The desired file path for the output TFLite model.
    """
    if not os.path.exists(onnx_model_path):
        print(f"Error: ONNX model not found at {onnx_model_path}")
        return

    # Define intermediate path for the TensorFlow SavedModel
    saved_model_path = "temp_tf_saved_model"

    try:
        print(f"Loading ONNX model from: {onnx_model_path}")
        # Load the ONNX model
        import onnx
        onnx_model = onnx.load(onnx_model_path)

        print(f"Converting ONNX model to TensorFlow SavedModel at: {saved_model_path}")
        # Convert ONNX model to TensorFlow SavedModel
        # The 'prepare' function creates a TensorFlow graph from the ONNX model
        # and can save it as a SavedModel.
        tf_rep = prepare(onnx_model)
        tf_rep.export_graph(saved_model_path)
        print("ONNX to TensorFlow SavedModel conversion complete.")

        print(f"Converting TensorFlow SavedModel to TFLite model at: {tflite_model_path}")
        # Convert the TensorFlow SavedModel to TFLite
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
        tflite_model = converter.convert()

        # Save the TFLite model
        with open(tflite_model_path, 'wb') as f:
            f.write(tflite_model)

        print(f"Successfully converted '{onnx_model_path}' to '{tflite_model_path}'")

    except Exception as e:
        print(f"An error occurred during conversion: {e}")
    finally:
        # Clean up the intermediate SavedModel directory
        if os.path.exists(saved_model_path):
            import shutil
            shutil.rmtree(saved_model_path)
            print(f"Cleaned up temporary directory: {saved_model_path}")

# --- Usage Example ---
if __name__ == "__main__":
    input_onnx_model = "models/best_ckpt.onnx"
    output_tflite_model = "models/best_ckpt.tflite"

    # Create the 'models' directory if it doesn't exist for the output
    os.makedirs(os.path.dirname(output_tflite_model), exist_ok=True)

    # Example: Create a dummy ONNX file for testing if it doesn't exist
    # In a real scenario, you would have your actual 'best_ckpt.onnx' file.
    if not os.path.exists(input_onnx_model):
        print(f"Dummy ONNX model '{input_onnx_model}' not found. Creating a placeholder for demonstration.")
        print("Please replace this with your actual ONNX model.")
        # This is a minimal valid ONNX graph for demonstration purposes.
        # It creates a simple graph with one input and one output.
        from onnx import helper, TensorProto
        from onnx import save

        # Define graph inputs
        X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 2, 3])
        # Define graph outputs
        Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 2, 3])
        # Define a simple node (e.g., Add)
        node_def = helper.make_node(
            'Add',
            inputs=['X', 'X'], # Adding X to itself
            outputs=['Y'],
        )
        # Define the graph
        graph_def = helper.make_graph(
            [node_def],
            'simple_graph',
            [X],
            [Y],
        )
        # Define the model
        model_def = helper.make_model(graph_def, producer_name='onnx-example')
        save(model_def, input_onnx_model)
        print(f"Dummy ONNX model created at: {input_onnx_model}")


    convert_onnx_to_tflite(input_onnx_model, output_tflite_model)


Loading ONNX model from: models/best_ckpt.onnx
Converting ONNX model to TensorFlow SavedModel at: temp_tf_saved_model
Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.NEAREST_NEIGHBOR...)` instead.


Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.NEAREST_NEIGHBOR...)` instead.
INFO:absl:Function `__call__` contains input name(s) x, y with unsupported characters which will be renamed to transpose_343_x, add_94_y in the SavedModel.
INFO:absl:Found untraced functions such as gen_tensor_dict while saving (showing 1 of 1). These functions will not be directly callable after loading.


INFO:tensorflow:Assets written to: temp_tf_saved_model\assets


INFO:tensorflow:Assets written to: temp_tf_saved_model\assets
INFO:absl:Writing fingerprint to temp_tf_saved_model\fingerprint.pb


ONNX to TensorFlow SavedModel conversion complete.
Converting TensorFlow SavedModel to TFLite model at: models/best_ckpt.tflite
Successfully converted 'models/best_ckpt.onnx' to 'models/best_ckpt.tflite'
Cleaned up temporary directory: temp_tf_saved_model


In [4]:
import tensorflow as tf

tflite_model_path = "models/best_ckpt.tflite"

try:
    # Load the TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print("\n--- Input Details ---")
    for i, detail in enumerate(input_details):
        print(f"Input {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")

    print("\n--- Output Details ---")
    for i, detail in enumerate(output_details):
        print(f"Output {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")

except Exception as e:
    print(f"Error inspecting TFLite model: {e}")


--- Input Details ---
Input 0:
  Name: serving_default_images:0
  Shape: [  1   3 640 640]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)

--- Output Details ---
Output 0:
  Name: PartitionedCall:0
  Shape: [   1 8400    7]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
